In [2]:
# 판다스 넘파이 가져오기
import pandas as pd
import numpy as np

In [3]:
# 파일 불러오기
order = pd.read_csv('./data/orders.csv')
oi = pd.read_csv('./data/order_items.csv')
prod = pd.read_csv('./data/products.csv')

In [4]:
prod.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    500 non-null    int64  
 1   product_name  500 non-null    str    
 2   category      500 non-null    str    
 3   price         500 non-null    str    
 4   cost          500 non-null    float64
dtypes: float64(1), int64(1), str(3)
memory usage: 35.5 KB


In [5]:
# orders 파일 정제
print(f'주문일시 결측 주문 건수 : {order["order_datetime"].isna().sum()}')
orders=order.dropna(subset='order_datetime')
print(f'주문일시 결측 제거 후 주문 건수 : {orders.shape[0]}')
orders=orders.drop_duplicates(subset='order_id')
print(f'중복 제거 후 주문건 수 : {orders.shape[0]}')
orders['order_datetime']=pd.to_datetime(orders['order_datetime'],errors="coerce")
orders.info()

주문일시 결측 주문 건수 : 1933
주문일시 결측 제거 후 주문 건수 : 198067
중복 제거 후 주문건 수 : 198067
<class 'pandas.DataFrame'>
Index: 198067 entries, 0 to 199999
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_id        198067 non-null  int64         
 1   customer_id     198067 non-null  int64         
 2   order_datetime  198067 non-null  datetime64[us]
 3   channel         198067 non-null  str           
 4   status          198067 non-null  str           
dtypes: datetime64[us](1), int64(2), str(2)
memory usage: 11.2 MB


In [6]:
#product함수 관련 결측치 및 제거
prod=prod.drop_duplicates()
prod['price']=pd.to_numeric(prod['price'].astype(str)
                            .str.replace(',','').str.replace('원',''))
prod= prod[prod['price']>0]
prod= prod[prod['cost']>0]
prod.info()

<class 'pandas.DataFrame'>
Index: 489 entries, 1 to 499
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    489 non-null    int64  
 1   product_name  489 non-null    str    
 2   category      489 non-null    str    
 3   price         489 non-null    float64
 4   cost          489 non-null    float64
dtypes: float64(2), int64(1), str(2)
memory usage: 35.2 KB


In [7]:
# order_item 파일 정제 (1, 주문 아이템 아이디 중복제거)
orderitem=oi.drop_duplicates(subset='order_item_id')
orderitem.info()

<class 'pandas.DataFrame'>
Index: 499880 entries, 0 to 499999
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   order_item_id  499880 non-null  int64  
 1   order_id       499880 non-null  int64  
 2   product_id     499880 non-null  int64  
 3   quantity       499880 non-null  int64  
 4   unit_price     484774 non-null  float64
 5   discount       499880 non-null  float64
dtypes: float64(2), int64(4)
memory usage: 26.7 MB


In [8]:
# order_item 파일 정제 (2. 유닛 프라이스 채우기)
orderitem= orderitem.merge(prod[['product_id','price']],how='left',on='product_id')
orderitem['unit_price']= orderitem['unit_price'].fillna(orderitem["price"])

orderitem=orderitem.drop_duplicates()
orderitem=orderitem.drop(columns='price')
orderitem.info()

<class 'pandas.DataFrame'>
RangeIndex: 499880 entries, 0 to 499879
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   order_item_id  499880 non-null  int64  
 1   order_id       499880 non-null  int64  
 2   product_id     499880 non-null  int64  
 3   quantity       499880 non-null  int64  
 4   unit_price     499641 non-null  float64
 5   discount       499880 non-null  float64
dtypes: float64(2), int64(4)
memory usage: 22.9 MB


In [9]:
# orders & order_items 결합
mix_ooi = orderitem.merge(orders,how='left',on='order_id',validate="m:1")
mix_ooi.info()

<class 'pandas.DataFrame'>
RangeIndex: 499880 entries, 0 to 499879
Data columns (total 10 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_item_id   499880 non-null  int64         
 1   order_id        499880 non-null  int64         
 2   product_id      499880 non-null  int64         
 3   quantity        499880 non-null  int64         
 4   unit_price      499641 non-null  float64       
 5   discount        499880 non-null  float64       
 6   customer_id     495100 non-null  float64       
 7   order_datetime  495100 non-null  datetime64[us]
 8   channel         495100 non-null  str           
 9   status          495100 non-null  str           
dtypes: datetime64[us](1), float64(3), int64(4), str(2)
memory usage: 43.5 MB


In [10]:
#주문정보가 존재하지 않는, 주문 품목 갯수 발견, 정확한 주문과 주문일시를 확인이 불가능하므로,
# 제거 예정
print(f'주문에 들어가있지 않은 주문 품목 수 : {mix_ooi["order_datetime"].isna().sum()}')
mix_ooi = mix_ooi.dropna(subset='order_datetime')
print(f'결측치 제거 후 총 주문품목수 : {mix_ooi.shape[0]}')

주문에 들어가있지 않은 주문 품목 수 : 4780
결측치 제거 후 총 주문품목수 : 495100


In [11]:
# 순매출을 구하기 위해서 prod와 병합(price 제외)
mixed = mix_ooi.merge(prod[['product_id','product_name','category','cost']],how='left',on='product_id',validate="m:1")
mixed.info()

<class 'pandas.DataFrame'>
RangeIndex: 495100 entries, 0 to 495099
Data columns (total 13 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_item_id   495100 non-null  int64         
 1   order_id        495100 non-null  int64         
 2   product_id      495100 non-null  int64         
 3   quantity        495100 non-null  int64         
 4   unit_price      494862 non-null  float64       
 5   discount        495100 non-null  float64       
 6   customer_id     495100 non-null  float64       
 7   order_datetime  495100 non-null  datetime64[us]
 8   channel         495100 non-null  str           
 9   status          495100 non-null  str           
 10  product_name    487853 non-null  str           
 11  category        487853 non-null  str           
 12  cost            487853 non-null  float64       
dtypes: datetime64[us](1), float64(4), int64(4), str(4)
memory usage: 66.3 MB


In [12]:
mixed['status'].unique()

<ArrowStringArray>
['delivered', 'paid', 'canceled', 'shipped', 'returned']
Length: 5, dtype: str

In [13]:
mixed['amount'] = (mixed['quantity'] * mixed['unit_price'])
mixed['p_amount'] = (mixed['quantity'] * mixed['unit_price'])*(1-mixed['discount'])
# 취소 환불 제거.
mixed_cr = mixed[(mixed['status'].isin(['canceled','returned']))]
mixed['status'].unique()

<ArrowStringArray>
['delivered', 'paid', 'canceled', 'shipped', 'returned']
Length: 5, dtype: str

In [14]:
mixed=mixed.set_index('order_datetime')
mixed_cr=mixed_cr.set_index('order_datetime')

In [17]:
amount_month = mixed.resample("ME").agg(매출=("amount","sum")
                                        ,순매출=("p_amount","sum")
                                        ,총주문품목수=("order_item_id","count")
                                        ,총주문수=("order_id","nunique"))
cancel_month = mixed_cr.resample("ME").agg(환불취소매출=("p_amount","sum")
                                           ,취소환불주문품목수=("order_item_id","count")
                                           ,취소환불주문수=("order_id","nunique"))
amount_month["취소환불제외액"]=amount_month["순매출"]-cancel_month["환불취소매출"]
amount_month["유효주문수"]=amount_month["총주문수"]-cancel_month["취소환불주문수"]
amount_month["유효주문품목수"]=amount_month["총주문품목수"]-cancel_month["취소환불주문품목수"]
amount_month=amount_month.loc["2024-01":"2024-06"]


In [18]:
amount_month

,매출,순매출,총주문품목수,총주문수,취소환불제외액,유효주문수,유효주문품목수
order_datetime,,,,,,,
2024-01-31,9.009169e+09,6.754357e+09,58986,21660,5.702378e+09,18411,50012
2024-02-29,9.838969e+09,7.377902e+09,64439,23697,6.266581e+09,20192,54812
2024-03-31,1.226160e+10,9.191785e+09,80791,29721,7.747580e+09,25140,68191
2024-04-30,1.315524e+10,9.877887e+09,85959,31591,8.379931e+09,26765,72777
2024-05-31,1.492853e+10,1.118529e+10,97404,35880,9.515943e+09,30515,83034
2024-06-30,1.630447e+10,1.223542e+10,107415,39492,1.041497e+10,33700,91539
